# Enriquecimento NPZ com 11 Canais Estruturais (Fase A.2)

**Spec**: `specs/004-melhoria-geracao-dados-cnn/` (T-A2-003).

Le cada NPZ produzido na Fase A.1 (Databricks), computa o tensor `canais (N, 4, 3, 11) int8` chamando `extrair_canais()` para cada estado, adiciona o array `nomes_canais (11,) U32` (auto-descricao) e regrava o NPZ via **sobrescrita atomica** (`tmp_path = path + '.tmp'` + `os.replace(...)`).

**Garantias** (NFR-06, FR-B-03):
- **Atomico**: nunca corrompe original mesmo com Ctrl+C — o `os.replace` so executa quando o `.tmp` esta integro.
- **Idempotente**: re-rodar sobre um NPZ ja enriquecido apenas confirma e regrava.
- **Preserva chaves**: `estados`, `rotulos`, `scores`, `generation_mode`, `labels_canonicos`, `minimax_depth` sao copiados sem modificacao.

**Pre-requisitos**:
1. Modulo `gerador_dados/jogo_pontinhos/analisador_estrutural_pontinhos.py` instalado (T-A2-001).
2. Diretorio com NPZs Fase A.1 montado (Databricks export ou copia local).

In [1]:
# Imports
import os
import sys
import glob
import time
import hashlib
import numpy as np

# Permitir importar o pacote do backend a partir do notebook (Colab/local).
ROOT = os.environ.get('ARENA_SAGAZ_BACKEND', os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from gerador_dados.jogo_pontinhos.analisador_estrutural_pontinhos import (
    NOMES_CANAIS,
    extrair_canais,
)

print('NOMES_CANAIS:', NOMES_CANAIS)
print('Backend root:', ROOT)

NOMES_CANAIS: ('aresta_topo', 'aresta_base', 'aresta_esquerda', 'aresta_direita', 'caixa_fechada', 'eh_grau3', 'eh_grau2', 'em_cadeia_curta', 'em_cadeia_longa', 'em_loop', 'em_cadeia_aberta_uma_ponta')
Backend root: d:\Desenvolvimento\arena-sagaz\arena-sagaz-backend


In [2]:
# Parametros do enriquecimento.

# Diretorio que contem os NPZ Fase A.1 (sem `canais`/`nomes_canais`).
# DIR_NPZ = os.environ.get(
#     'DIR_NPZ',
#     '/dados/profundidade_minimax_11_v7_adaptativo',
# )
DIR_NPZ = os.path.join(ROOT, 'dados', 'profundidade_minimax_11_v7_adaptativo')

# Padrao do nome dos arquivos.
PADRAO = 'dataset_pequeno_*.npz'

# Se True, o notebook regrava mesmo arquivos que ja contem 'canais'
# (idempotencia explicita); util quando o algoritmo de canais muda.
FORCAR_REGRAVAR = True

print(f'DIR_NPZ        = {DIR_NPZ}')
print(f'PADRAO         = {PADRAO}')
print(f'FORCAR_REGRAVAR= {FORCAR_REGRAVAR}')

DIR_NPZ        = d:\Desenvolvimento\arena-sagaz\arena-sagaz-backend\dados\profundidade_minimax_11_v7_adaptativo
PADRAO         = dataset_pequeno_*.npz
FORCAR_REGRAVAR= True


In [3]:
# Funcao auxiliar: regrava NPZ enriquecido com sobrescrita atomica.

def enriquecer_arquivo(caminho: str, *, forcar: bool = False) -> dict:
    """Le NPZ Fase A.1, computa canais, regrava enriquecido (atomico).

    Returns:
        dict com chaves: 'caminho', 'n_estados', 'tinha_canais', 'tempo_s',
        'hash_estados_in', 'hash_estados_out'.
    """
    t0 = time.time()
    with np.load(caminho, allow_pickle=True) as npz_file:
        d = dict(npz_file)
        
    tinha_canais = 'canais' in d
    if tinha_canais and not forcar:
        return {
            'caminho': caminho,
            'n_estados': int(d['estados'].shape[0]),
            'tinha_canais': True,
            'tempo_s': 0.0,
            'hash_estados_in': hashlib.sha256(d['estados'].tobytes()).hexdigest()[:16],
            'hash_estados_out': hashlib.sha256(d['estados'].tobytes()).hexdigest()[:16],
        }

    estados = d['estados']
    n = estados.shape[0]
    canais = np.zeros((n, 4, 3, 11), dtype=np.int8)
    for i in range(n):
        canais[i] = extrair_canais(estados[i])

    nomes_canais = np.array(NOMES_CANAIS, dtype='U32')

    # Monta novo dict preservando todas as chaves originais.
    novo = dict(d)
    novo['canais'] = canais
    novo['nomes_canais'] = nomes_canais

    # Sobrescrita atomica: salva em .tmp, depois os.replace().
    tmp_path = caminho + '.tmp.npz'
    np.savez_compressed(tmp_path, **novo)
    os.replace(tmp_path, caminho)

    return {
        'caminho': caminho,
        'n_estados': n,
        'tinha_canais': tinha_canais,
        'tempo_s': time.time() - t0,
        'hash_estados_in': hashlib.sha256(estados.tobytes()).hexdigest()[:16],
        'hash_estados_out': hashlib.sha256(estados.tobytes()).hexdigest()[:16],
    }

print('Funcao enriquecer_arquivo() pronta.')

Funcao enriquecer_arquivo() pronta.


In [4]:
# Loop principal: enriquece todos os NPZs do diretorio.

arquivos = sorted(glob.glob(os.path.join(DIR_NPZ, PADRAO)))
print(f'Encontrados {len(arquivos)} arquivos.')

relatorio = []
for arq in arquivos:
    info = enriquecer_arquivo(arq, forcar=FORCAR_REGRAVAR)
    print(
        f"{os.path.basename(info['caminho'])}: "
        f"N={info['n_estados']} "
        f"tinha_canais={info['tinha_canais']} "
        f"t={info['tempo_s']:.2f}s"
    )
    relatorio.append(info)

print()
print(f'Total de arquivos processados: {len(relatorio)}')
print(f'Tempo total: {sum(r["tempo_s"] for r in relatorio):.1f}s')

Encontrados 152 arquivos.
dataset_pequeno_0001.npz: N=5000 tinha_canais=True t=0.30s
dataset_pequeno_0002.npz: N=5000 tinha_canais=True t=0.29s
dataset_pequeno_0003.npz: N=5000 tinha_canais=True t=0.29s
dataset_pequeno_0004.npz: N=5000 tinha_canais=True t=0.29s
dataset_pequeno_0005.npz: N=5000 tinha_canais=True t=0.29s
dataset_pequeno_0006.npz: N=5000 tinha_canais=True t=0.30s
dataset_pequeno_0007.npz: N=5000 tinha_canais=True t=0.29s
dataset_pequeno_0008.npz: N=5000 tinha_canais=True t=0.29s
dataset_pequeno_0009.npz: N=5000 tinha_canais=True t=0.29s
dataset_pequeno_0010.npz: N=5000 tinha_canais=True t=0.29s
dataset_pequeno_0011.npz: N=5000 tinha_canais=True t=0.29s
dataset_pequeno_0012.npz: N=5000 tinha_canais=True t=0.30s
dataset_pequeno_0013.npz: N=5000 tinha_canais=True t=0.29s
dataset_pequeno_0014.npz: N=5000 tinha_canais=True t=0.29s
dataset_pequeno_0015.npz: N=5000 tinha_canais=True t=0.30s
dataset_pequeno_0016.npz: N=5000 tinha_canais=True t=0.30s
dataset_pequeno_0017.npz: N=50

In [5]:
# Validacao pos-enriquecimento: nomes_canais byte-a-byte identico em todos os arquivos.

esperado = np.array(NOMES_CANAIS, dtype='U32')
ok = True
for arq in arquivos:
    d = np.load(arq, allow_pickle=True)
    if 'canais' not in d.files:
        print(f'ERRO {arq}: campo `canais` ausente.')
        ok = False
        continue
    if 'nomes_canais' not in d.files:
        print(f'ERRO {arq}: campo `nomes_canais` ausente.')
        ok = False
        continue
    if not np.array_equal(d['nomes_canais'], esperado):
        print(f'ERRO {arq}: nomes_canais divergente.')
        print(f'  obtido:   {list(d["nomes_canais"])}')
        print(f'  esperado: {list(esperado)}')
        ok = False
        continue
    canais = d['canais']
    if canais.dtype != np.int8 or canais.shape[1:] != (4, 3, 11):
        print(f'ERRO {arq}: shape/dtype canais invalido ({canais.shape} {canais.dtype}).')
        ok = False
        continue
    if set(np.unique(canais).tolist()) - {0, 1}:
        print(f'ERRO {arq}: canais fora de {{0,1}}.')
        ok = False
        continue

if ok:
    print('OK: todos os NPZs validados.')
else:
    print('FALHA: ver mensagens acima.')

OK: todos os NPZs validados.
